# Modélisation — Random Forest

Pipeline complet pour la prédiction de la gravité des accidents de la route (2024).

**Étapes :** chargement des données prétraitées · entraînement · évaluation · validation croisée · importance des variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Chargement des Données Prétraitées

In [ ]:
processed_dir = Path.cwd().parent / 'data' / 'processed'

X_train = pd.read_csv(processed_dir / 'X_train.csv')
X_test  = pd.read_csv(processed_dir / 'X_test.csv')
y_train = pd.read_csv(processed_dir / 'y_train.csv').squeeze()
y_test  = pd.read_csv(processed_dir / 'y_test.csv').squeeze()

print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(y_train.value_counts())

## 2. Vérification du Jeu de Données

In [ ]:
# Les fichiers prétraités ont déjà été nettoyés et encodés par 01_preparation_donnees.ipynb
print(f'Nombre de features : {X_train.shape[1]}')
print(f'Distribution des classes (train) :')
print(y_train.value_counts())

## 3. Répartition Train / Test

In [ ]:
# Les données sont déjà découpées et encodées — pas de split supplémentaire nécessaire
print(f'Taille train : {X_train.shape[0]} échantillons')
print(f'Taille test  : {X_test.shape[0]} échantillons')
print(f'Proportion Tué (train) : {(y_train == "Tué").mean():.2%}')

## 4. Modélisation avec Random Forest Classifier

### Stratégie de compensation du déséquilibre

La classe « Tué » représente seulement ~7 % des données. Un poids ×20 lui est attribué afin de maximiser le **Recall** sur cette classe critique.

In [ ]:
# Poids sur-représentant la classe "Tué" (×20) pour maximiser le Recall sur la classe rare
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight={'Blessé léger': 1, 'Blessé hospitalisé': 2, 'Tué': 20},
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('Modèle entraîné.')

## 5. Évaluation du Modèle

### Métriques de performance
- **Accuracy** : proportion de prédictions correctes
- **Recall** : proportion de vrais positifs détectés *(métrique prioritaire pour « Tué »)*
- **F1-Score** : moyenne harmonique précision/recall

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred, label=''):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'{label}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    return acc, prec, rec, f1

evaluate_model(y_test, y_pred, 'Random Forest — Ensemble de test')
print()
print(classification_report(y_test, y_pred))

## 6. Matrice de Confusion

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

## 7. Validation Croisée (5-Fold Stratifié)

La validation croisée stratifiée garantit la représentation de chaque classe dans chaque pli, ce qui est essentiel avec des classes déséquilibrées.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='recall_macro')

print(f'Recall macro — CV 5-fold : {scores.mean():.3f} (+/- {scores.std():.3f})')
print(f'Scores par fold : {scores.round(3)}')

## 8. Importance des Variables

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 10 des variables les plus importantes :')
print(feature_importance.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
top10 = feature_importance.head(10)
ax.barh(top10['Feature'][::-1], top10['Importance'][::-1], color='steelblue')
ax.set_title('Top 10 — Importance des Variables (Random Forest)', fontweight='bold')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

## 9. Résumé et Recommandations

In [ ]:
print('=' * 65)
print('RÉSUMÉ — RANDOM FOREST')
print('=' * 65)
print(f'  n_estimators : {rf.n_estimators}')
print(f'  class_weight : Tué ×20, Blessé hospitalisé ×2')
print(f'  Recall macro CV : {scores.mean():.3f}')
print()
print('  Avantages : robustesse, gestion du déséquilibre, importance des variables.')
print('  Limites   : temps d\'entraînement élevé, interprétabilité réduite.')